# Trending Tweet Tracker

## Problem Statement

A social platform tracks each user's daily tweet count.

For every user and date, return the average tweet count across that date and the two preceding available rows for the same user.

At the beginning of a user's history, use only the available rows when calculating the average.

## Input Table

### ttt_tweet_input

| Column Name | Data Type |
|------------|-----------|
| user_id | INT |
| tweet_date | TIMESTAMP |
| tweet_count | INT |

## Requirements

- Calculate results independently for each user.
- Order records within each user by `tweet_date` ascending.
- Use the current row and up to two preceding available rows for the same user.
- Round the calculated value to 2 decimal places.
- Return results ordered by `user_id` and `tweet_date` in ascending order.
- Return results matching the required schema and order.

## Output Columns

| Column Name |
|------------|
| user_id |
| tweet_date |
| rolling_avg_3d |

## Sample Input

### ttt_tweet_input

| user_id | tweet_date | tweet_count |
|---------|------------|-------------|
| 111 | 2022-06-01T00:00:00 | 2 |
| 111 | 2022-06-02T00:00:00 | 1 |
| 111 | 2022-06-03T00:00:00 | 3 |
| 111 | 2022-06-04T00:00:00 | 4 |
| 111 | 2022-06-05T00:00:00 | 5 |

## Expected Output Schema

| Column Name | Data Type |
|------------|-----------|
| user_id | INT |
| tweet_date | DATE |
| rolling_avg_3d | DECIMAL |

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.window import Window

ttt_tweet_input_schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("tweet_date", TimestampType(), True),
    StructField("tweet_count", IntegerType(), True)
])

ttt_tweet_input_data = [
    (111, "2022-06-01 00:00:00", 2),
    (111, "2022-06-02 00:00:00", 1),
    (111, "2022-06-03 00:00:00", 3),
    (111, "2022-06-04 00:00:00", 4),
    (111, "2022-06-05 00:00:00", 5)
]

ttt_tweet_input_df = spark.createDataFrame(
    ttt_tweet_input_data,
    ["user_id", "tweet_date", "tweet_count"]
).withColumn(
    "tweet_date",
    to_timestamp(col("tweet_date"))
)

In [0]:
result_df = (
    ttt_tweet_input_df.withColumn(
        "avg_tweet_count",
        avg(col("tweet_count")).over(
            Window.partitionBy("user_id").orderBy(col("tweet_date")).rowsBetween(-2, 0)
        ),
    )
    .select(
        col("user_id"),
        date_format(col("tweet_date"), "yyyy-MM-dd").alias("tweet_date"),
        round(col("avg_tweet_count"), 2).alias("avg_tweet_count"),
    )
    .orderBy(col("user_id"), col("tweet_date"))
)
display(result_df)